# 25 — Ronda final de ideas: precio compartido (R0), limpieza de etiquetas (R1), propagación por vendedor (R2), conteos sobre train + test (R3)

Ideas surgidas al pasarle a otra IA un resumen con el contexto completo del TP. Todas se miden igual que siempre:
- **filtro rápido:** HistGB P7a con duración (0,9412), comparación pareada fold por fold (pasa con 5/5, o 4/5 y media > desvío);
- **confirmación** de lo que pase en el promedio final (CatBoost + HistGB + duración, 0,9420), con las semillas 42 y 7.

| | Qué ataca | ¿En el material? |
|---|---|---|
| R0 | Precios "de referencia": el mismo precio exacto en la misma categoría, usado por muchos vendedores distintos | *Frequency encoder* / *feature engineering* (`Copia de AP_Categóricas.ipynb`) |
| R1 | El ≈ 1,4% de etiquetas que contradicen el título (`08_errores`): si el modelo aprende de etiquetas mal puestas, aprende reglas equivocadas | **No** (*confident learning*). Usa probabilidades *out-of-fold* (`S8_Ensambles.ipynb`, `cross_val_predict`) |
| R2 | Vendedores sin historial en train (la mitad de los errores): sus otras publicaciones del test sí se conocen, aunque sin etiqueta | **No** como técnica (es parecido a Stacking/*pseudo-labeling*) |
| R3 | Tamaño "real" del vendedor, la categoría y el título contando también el test | *Frequency encoder* (`Copia de AP_Categóricas.ipynb`) |

In [1]:
import sys
sys.path.append("..")
import os, re, time, warnings, unicodedata
import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
crudo_test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = np.empty(len(train), dtype=int)
for k, (_, v) in enumerate(cv.split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)

# Referencias de la 18 (semilla 42), las mismas de la 22-24
oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
assert (oof_ref["fold"].to_numpy() == folds).all()
prob_ref = oof_ref["histgb"].to_numpy()          # HistGB P7a + duración, out-of-fold


def acc_por_fold(prob, filas=None):
    filas = np.ones(len(y_np), bool) if filas is None else filas
    acierto = (np.asarray(prob) >= 0.5).astype(int) == y_np
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


acc_ref = acc_por_fold(prob_ref)
print("HistGB P7a + duración:", acc_ref.round(4), acc_ref.mean().round(4))


def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def p7a_con(extras, cacheado=True):
    pipeline = armar_pipeline_histgb_combos()
    if cacheado:
        pipeline = con_texto_cacheado(pipeline)
    pipeline.named_steps["preprocesador"].transformers.extend(extras)
    return pipeline


resultados = {}

HistGB P7a + duración: [0.9428 0.9393 0.9426 0.9394 0.9422] 0.9412


## R0 — Precio "de referencia" compartido entre vendedores

**EDA (train; el conteo es sobre train + test):** agrupando por (`category_id`, `price` exacto), el % de `new` baja de **64%** cuando el precio es único en su categoría a **16%** cuando más de 20 vendedores distintos usan ese mismo precio en esa categoría. El 33% de las filas comparte el par con al menos otro vendedor (test: igual). **Interpretación (no verificada):** los usados se publican a precios "de mercado" que todos copian (el precio de referencia de un celular usado), mientras que los nuevos de tienda tienen precios propios (con decimales, márgenes, etc.).

**Features** (en log1p, aunque a un árbol el log no le cambia los cortes):
- `n_vendedores_mismo_precio_cat`: vendedores **distintos** con el mismo (`category_id`, `price`);
- `n_publicaciones_mismo_precio_cat`: publicaciones con ese par;
- `n_vendedores_mismo_precio_global`: vendedores distintos con el mismo `price`, en cualquier categoría.

Se usa el **precio crudo** (sin capear): el capeo del `Pipeline` lleva el 0,5% más caro a un mismo techo y eso inventaría "precios compartidos".

**Dos variantes:**
- **R0a (train + test, determinística, afuera del `Pipeline`):** cuenta sobre las 100.000 filas. **No es leakage:** solo cuenta filas y vendedores, nunca usa `condition`; una fila de validación y una de test se cuentan igual (las dos están en el universo). **Limitación para producción:** es transductiva, igual que R3: necesita tener el lote completo de antemano.
- **R0b (solo el train del fold, transformer adentro del `Pipeline`):** cuenta solo con las filas de entrenamiento. Sirve para ver **cuánto depende de usar el test**, y es la versión que funcionaría en producción (se cuenta contra el histórico). Detalle: en train cada fila ya está dentro del conteo de su par; una fila de validación/test no. Para que se describan igual, en `transform` cada fila se **suma a sí misma** (+1 publicación, y +1 vendedor si su vendedor no tenía ya ese par).

**Control de redondez:** el bloque M (`07_nuevas_features`: `tiene_decimales`, `mult_100`, `mult_1000`, `termina_9`, `termina_99`, sobre el precio crudo) no había pasado en `11b_marginales`. Si R0 fuera solo "precio redondo", su aporte desaparecería sobre un modelo que ya tiene esas 5 flags. Por eso R0 también se mide contra **P7a + duración + M**.

In [2]:
train["precio_crudo"] = crudo["price"].to_numpy()          # sin capear (el capeo solo modifica price/base_price)
R0_COLS = ["n_vendedores_mismo_precio_cat", "n_publicaciones_mismo_precio_cat", "n_vendedores_mismo_precio_global"]

# R0a: conteo sobre train + test (determinístico)
universo_precio = pd.DataFrame({"vendedor": np.r_[crudo["seller_id"], crudo_test["seller_id"]],
                                "categoria": np.r_[crudo["category_id"], crudo_test["category_id"]],
                                "precio": np.r_[crudo["price"], crudo_test["price"]]})
g_cat = universo_precio.groupby(["categoria", "precio"])["vendedor"]
r0a = pd.DataFrame({
    "r0a_" + R0_COLS[0]: np.log1p(g_cat.transform("nunique")),
    "r0a_" + R0_COLS[1]: np.log1p(g_cat.transform("size")),
    "r0a_" + R0_COLS[2]: np.log1p(universo_precio.groupby("precio")["vendedor"].transform("nunique")),
})
for c in r0a:
    train[c] = r0a[c].to_numpy()[:len(train)]

# Bloque M (07_nuevas_features), sobre el precio crudo
M_COLS = ["tiene_decimales", "mult_100", "mult_1000", "termina_9", "termina_99"]
precio = train["precio_crudo"]
train["tiene_decimales"] = (precio % 1 != 0).astype(int)
train["mult_100"] = (precio % 100 == 0).astype(int)
train["mult_1000"] = (precio % 1000 == 0).astype(int)
train["termina_9"] = (precio.astype("int64") % 10 == 9).astype(int)
train["termina_99"] = (precio.astype("int64") % 100 == 99).astype(int)

# EDA: % de new por cantidad de vendedores con el mismo (categoría, precio), y cruce con redondez
n_vend = np.expm1(train["r0a_" + R0_COLS[0]]).round().astype(int)
tramo = pd.cut(n_vend, [0, 1, 2, 5, 20, np.inf], labels=["1", "2", "3-5", "6-20", "> 20"])
n_vend_test = g_cat.transform("nunique").to_numpy()[len(train):]
print(pd.DataFrame({"filas_train": tramo.value_counts().sort_index(), "pct_train": tramo.value_counts(normalize=True).sort_index(),
                    "pct_test": pd.cut(pd.Series(n_vend_test), [0, 1, 2, 5, 20, np.inf], labels=["1", "2", "3-5", "6-20", "> 20"]).value_counts(normalize=True).sort_index(),
                    "pct_new": y.groupby(tramo, observed=False).mean()}).round(3))
print("\n% de new: múltiplo de 100 (filas) × precio compartido con otro vendedor (columnas)")
cruce = [train["mult_100"], (n_vend > 1).rename("compartido")]
print(pd.crosstab(*cruce, values=y, aggfunc="mean").round(3)); print(pd.crosstab(*cruce))

      filas_train  pct_train  pct_test  pct_new
1           46929      0.670     0.670    0.641
2            7597      0.109     0.105    0.456
3-5          6779      0.097     0.098    0.345
6-20         5487      0.078     0.081    0.215
> 20         3208      0.046     0.046    0.158

% de new: múltiplo de 100 (filas) × precio compartido con otro vendedor (columnas)
compartido  False  True 
mult_100                
0           0.714  0.332
1           0.374  0.301
compartido  False  True 
mult_100                
0           36945  17595
1            9984   5476


In [3]:
class PrecioCompartido(BaseEstimator, TransformerMixin):
    # Conteos de R0 aprendidos SOLO con las filas del fit (train del fold). Columnas: seller_id, category_id, precio_crudo
    def fit(self, X, y=None):
        self.nv_cat_ = X.groupby(["category_id", "precio_crudo"])["seller_id"].nunique()
        self.np_cat_ = X.groupby(["category_id", "precio_crudo"]).size()
        self.nv_glob_ = X.groupby("precio_crudo")["seller_id"].nunique()
        self.trio_ = set(zip(X["seller_id"], X["category_id"], X["precio_crudo"]))
        self.par_ = set(zip(X["seller_id"], X["precio_crudo"]))
        return self

    def transform(self, X):
        # Filas que no participaron del fit (validación/test): se suman a sí mismas
        clave = pd.MultiIndex.from_arrays([X["category_id"], X["precio_crudo"]])
        nuevo_trio = np.array([t not in self.trio_ for t in zip(X["seller_id"], X["category_id"], X["precio_crudo"])])
        nuevo_par = np.array([p not in self.par_ for p in zip(X["seller_id"], X["precio_crudo"])])
        nv = self.nv_cat_.reindex(clave).fillna(0).to_numpy() + nuevo_trio
        npub = self.np_cat_.reindex(clave).fillna(0).to_numpy() + 1
        nvg = self.nv_glob_.reindex(X["precio_crudo"]).fillna(0).to_numpy() + nuevo_par
        return np.log1p(np.c_[nv, npub, nvg])

    def fit_transform(self, X, y=None):
        # Filas del fit: ya están dentro de su conteo
        self.fit(X)
        g = X.groupby(["category_id", "precio_crudo"])["seller_id"]
        return np.log1p(np.c_[g.transform("nunique"), g.transform("size"),
                              X.groupby("precio_crudo")["seller_id"].transform("nunique")])

    def get_feature_names_out(self, input_features=None):
        return np.array(["r0b_" + c for c in R0_COLS])


# Verificación: un mismo par visto en train por 2 vendedores (A, B) y una fila nueva de C en validación
ej_tr = pd.DataFrame({"seller_id": ["A", "B", "A"], "category_id": ["x", "x", "y"], "precio_crudo": [100.0, 100.0, 50.0]})
t = PrecioCompartido()
print("train:", np.expm1(t.fit_transform(ej_tr)).round(0).tolist())
print("val (C en x/100, A en x/100, C en z/7):",
      np.expm1(t.transform(pd.DataFrame({"seller_id": ["C", "A", "C"], "category_id": ["x", "x", "z"],
                                         "precio_crudo": [100.0, 100.0, 7.0]}))).round(0).tolist())

train: [[2.0, 2.0, 2.0], [2.0, 2.0, 2.0], [1.0, 1.0, 1.0]]
val (C en x/100, A en x/100, C en z/7): [[3.0, 3.0, 3.0], [2.0, 3.0, 2.0], [1.0, 1.0, 1.0]]


In [4]:
def medir_extras(nombre, extras, ref, ref_nombre, sola_cols=None):
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sola = (cross_validate(HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED),
                               train[sola_cols], y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"].mean()
                if sola_cols else np.nan)
        acc = cross_validate(p7a_con(extras), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
    fila = {"señal_sola": sola, "con_P7a": acc.mean(), **comparar(acc, ref), "minutos": (time.time() - inicio) / 60,
            "referencia": ref_nombre}
    print(f"{nombre:52s} sola {sola:.4f} | {acc.mean():.4f} | dif {fila['dif_media']:+.4f} ({fila['folds_gana']}/5) vs {ref_nombre}")
    return fila, acc


r0a_cols = ["r0a_" + c for c in R0_COLS]
for c in r0a_cols:
    resultados[f"R0a {c[4:]}"], _ = medir_extras(f"R0a {c[4:]}", [("r0", "passthrough", [c])], acc_ref, "P7a", [c])
resultados["R0a (las 3 juntas)"], acc_r0a = medir_extras("R0a (las 3 juntas)", [("r0", "passthrough", r0a_cols)], acc_ref, "P7a", r0a_cols)
resultados["R0b (las 3, solo train del fold)"], acc_r0b = medir_extras(
    "R0b (las 3, solo train del fold)", [("r0", PrecioCompartido(), ["seller_id", "category_id", "precio_crudo"])], acc_ref, "P7a")

# Control de redondez: referencia = P7a + M
resultados["M (5 flags de redondez)"], acc_m = medir_extras("M (5 flags de redondez)", [("m", "passthrough", M_COLS)], acc_ref, "P7a")
resultados["R0a + M (vs P7a + M)"], _ = medir_extras("R0a + M (vs P7a + M)", [("m", "passthrough", M_COLS), ("r0", "passthrough", r0a_cols)],
                                                   acc_m, "P7a + M")
resultados["R0b + M (vs P7a + M)"], _ = medir_extras("R0b + M (vs P7a + M)", [("m", "passthrough", M_COLS),
                                                   ("r0", PrecioCompartido(), ["seller_id", "category_id", "precio_crudo"])], acc_m, "P7a + M")
pd.DataFrame(resultados).T.round(4)

R0a n_vendedores_mismo_precio_cat                    sola 0.6526 | 0.9409 | dif -0.0003 (2/5) vs P7a


R0a n_publicaciones_mismo_precio_cat                 sola 0.6371 | 0.9409 | dif -0.0003 (1/5) vs P7a


R0a n_vendedores_mismo_precio_global                 sola 0.6960 | 0.9409 | dif -0.0003 (2/5) vs P7a


R0a (las 3 juntas)                                   sola 0.7051 | 0.9409 | dif -0.0003 (2/5) vs P7a


R0b (las 3, solo train del fold)                     sola nan | 0.9412 | dif -0.0001 (2/5) vs P7a


M (5 flags de redondez)                              sola nan | 0.9416 | dif +0.0003 (2/5) vs P7a


R0a + M (vs P7a + M)                                 sola nan | 0.9411 | dif -0.0004 (2/5) vs P7a + M


R0b + M (vs P7a + M)                                 sola nan | 0.9416 | dif +0.0001 (3/5) vs P7a + M


,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos,referencia
R0a n_vendedores_mismo_precio_cat,0.6526,0.940943,-0.0003,0.000769,2,False,0.64473,P7a
R0a n_publicaciones_mismo_precio_cat,0.6371,0.940943,-0.0003,0.000506,1,False,0.571504,P7a
R0a n_vendedores_mismo_precio_global,0.696043,0.9409,-0.000343,0.00159,2,False,0.634353,P7a
R0a (las 3 juntas),0.7051,0.940943,-0.0003,0.000916,2,False,0.609825,P7a
"R0b (las 3, solo train del fold)",NaN,0.941186,-0.000057,0.001086,2,False,0.628787,P7a
M (5 flags de redondez),NaN,0.941571,0.000329,0.001024,2,False,0.601331,P7a
R0a + M (vs P7a + M),NaN,0.941143,-0.000429,0.001029,2,False,0.598821,P7a + M
R0b + M (vs P7a + M),NaN,0.941629,0.000057,0.000722,3,False,0.588769,P7a + M


## Pieza común a R1 y R2: predicciones *out-of-fold* internas

Para cada fold externo k (el que se evalúa), **solo con el train de ese fold** (los otros 4), se hace una CV interna de 5 y se obtiene, para cada fila de ese train, la probabilidad de `new` de un HistGB P7a + duración que **no la vio** (igual que `cross_val_predict` en `S8_Ensambles.ipynb`). Además se guarda en qué fold interno ("lote") cayó cada fila.

**No es leakage:** el fold externo k no participa en nada; sus filas no aportan ni features ni etiquetas. Son 25 ajustes del HistGB, y se guardan en `.cache_laboratorio/oof_interna_p7a_duracion.csv` para no repetirlos.

In [5]:
archivo_interna = os.path.join(carpeta_lab, "oof_interna_p7a_duracion.csv")
if os.path.exists(archivo_interna):
    interna = pd.read_csv(archivo_interna)
else:
    inicio, partes = time.time(), []
    for k in range(1, 6):
        filas_tr = np.where(folds != k)[0]
        cv_interna = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        lote = np.empty(len(filas_tr), dtype=int)
        for j, (_, v) in enumerate(cv_interna.split(filas_tr, y_np[filas_tr])):
            lote[v] = j + 1
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            prob = cross_val_predict(armar_pipeline_histgb_combos(), train.iloc[filas_tr], y.iloc[filas_tr],
                                     cv=cv_interna, method="predict_proba", n_jobs=5)[:, 1]
        partes.append(pd.DataFrame({"fold_externo": k, "fila": filas_tr, "lote_interno": lote, "prob": prob}))
        print(f"fold externo {k}: {(time.time() - inicio) / 60:.1f} min")
        with open(os.path.join(carpeta_lab, "ideas_finales.log"), "a") as f:
            f.write(f"OOF interna, fold externo {k}: {(time.time() - inicio) / 60:.1f} min\n")
    interna = pd.concat(partes, ignore_index=True)
    interna.to_csv(archivo_interna, index=False)
print("Accuracy interna (por fold externo):",
      interna.assign(ok=lambda d: (d["prob"] >= 0.5).astype(int) == y_np[d["fila"]]).groupby("fold_externo")["ok"].mean().round(4).tolist())

Accuracy interna (por fold externo): [0.9384, 0.9395, 0.9387, 0.9401, 0.9387]


## R1 — Limpieza de ruido de etiqueta (*confident learning*)

**Qué es (no está en el material).** Si una fila está mal etiquetada (un "Celular nuevo en caja" etiquetado `used`), el modelo intenta aprenderla igual, y eso empuja sus reglas en la dirección equivocada. *Confident learning* (Northcutt et al., 2021) propone:
1. obtener para cada fila de entrenamiento la probabilidad que le da un modelo que **no la vio** (*out-of-fold*);
2. calcular P(clase etiquetada): si la etiqueta es `new`, P(new); si es `used`, 1 − P(new);
3. marcar como **sospechosas** las filas donde esa probabilidad es muy baja (< umbral): el modelo, sin haberlas visto, está muy seguro de lo contrario;
4. **reentrenar sin las sospechosas**.

Ejemplo: una fila etiquetada `used` con P(new) = 0,98 tiene P(clase etiquetada) = 0,02. Con umbral 0,05 se saca; con P(new) = 0,90 (0,10) se queda.

**Por qué no es leakage:**
- las sospechosas se eligen **solo dentro del train del fold**, con las probabilidades internas de arriba;
- el fold de validación **no se toca**: se evalúa completo, con sus etiquetas originales, incluidas las que estén mal (igual que el test real, que no se puede limpiar);
- el umbral se elige **con 4 folds y se evalúa en el quinto**, rotando (además de reportar cada umbral fijo). La opción "no limpiar" (umbral 0) también es candidata.

**Riesgo:** sacar casos difíciles pero bien etiquetados (por ejemplo, "Departamento a estrenar" `new`, que en `08_errores` resultó estar bien). Por eso se muestran 20 ejemplos.

In [6]:
umbrales = [0.05, 0.10, 0.20]


def entrenar_limpio(k, umbral):
    d = interna[interna["fold_externo"] == k]
    p_etiqueta = np.where(y_np[d["fila"]] == 1, d["prob"], 1 - d["prob"])
    quedan = d["fila"].to_numpy()[p_etiqueta >= umbral]
    filas_va = np.where(folds == k)[0]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar_pipeline_histgb_combos().fit(train.iloc[quedan], y.iloc[quedan])
        prob = pipeline.predict_proba(train.iloc[filas_va])[:, 1]
    return k, umbral, len(d) - len(quedan), filas_va, prob


inicio = time.time()
salidas = Parallel(n_jobs=5)(delayed(entrenar_limpio)(k, u) for u in umbrales for k in range(1, 6))
minutos_r1 = (time.time() - inicio) / 60

prob_r1 = {u: np.empty(len(train)) for u in umbrales}
sacadas = pd.DataFrame(index=range(1, 6), columns=umbrales, dtype=float)
for k, u, n_sacadas, filas_va, prob in salidas:
    prob_r1[u][filas_va] = prob
    sacadas.loc[k, u] = n_sacadas
print("Filas sacadas por fold (de ~56.000):"); print(sacadas.astype(int))
print("% sacado:", (sacadas.mean() / 56000 * 100).round(2).to_dict())

acc_r1 = pd.DataFrame({0.0: acc_ref, **{u: acc_por_fold(prob_r1[u]) for u in umbrales}}, index=range(1, 6))
print("\nAccuracy por fold y umbral (0 = sin limpiar):"); print(acc_r1.round(4))
for u in umbrales:
    resultados[f"R1 umbral fijo {u}"] = {"con_P7a": acc_r1[u].mean(), **comparar(acc_r1[u].to_numpy(), acc_ref), "minutos": minutos_r1}

# Umbral elegido con 4 folds y evaluado en el quinto
elegido = {k: acc_r1.drop(index=k).mean().idxmax() for k in range(1, 6)}
acc_elegido = np.array([acc_r1.loc[k, elegido[k]] for k in range(1, 6)])
print("Umbral elegido en los otros 4 folds:", elegido)
resultados["R1 umbral elegido en 4 folds"] = {"con_P7a": acc_elegido.mean(), **comparar(acc_elegido, acc_ref), "minutos": minutos_r1}
pd.DataFrame(resultados).T.round(4)

Filas sacadas por fold (de ~56.000):
   0.05  0.10  0.20
1   396   824  1551
2   402   825  1450
3   404   831  1533
4   408   811  1479
5   415   811  1479
% sacado: {0.05: 0.72, 0.1: 1.46, 0.2: 2.68}

Accuracy por fold y umbral (0 = sin limpiar):
     0.00    0.05    0.10    0.20
1  0.9428  0.9419  0.9413  0.9411
2  0.9393  0.9378  0.9381  0.9351
3  0.9426  0.9409  0.9416  0.9408
4  0.9394  0.9407  0.9409  0.9384
5  0.9422  0.9420  0.9420  0.9403
Umbral elegido en los otros 4 folds: {1: np.float64(0.0), 2: np.float64(0.0), 3: np.float64(0.0), 4: np.float64(0.0), 5: np.float64(0.0)}


,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos,referencia
R0a n_vendedores_mismo_precio_cat,0.6526,0.940943,-0.0003,0.000769,2,False,0.64473,P7a
R0a n_publicaciones_mismo_precio_cat,0.6371,0.940943,-0.0003,0.000506,1,False,0.571504,P7a
R0a n_vendedores_mismo_precio_global,0.696043,0.9409,-0.000343,0.00159,2,False,0.634353,P7a
R0a (las 3 juntas),0.7051,0.940943,-0.0003,0.000916,2,False,0.609825,P7a
"R0b (las 3, solo train del fold)",NaN,0.941186,-0.000057,0.001086,2,False,0.628787,P7a
M (5 flags de redondez),NaN,0.941571,0.000329,0.001024,2,False,0.601331,P7a
R0a + M (vs P7a + M),NaN,0.941143,-0.000429,0.001029,2,False,0.598821,P7a + M
R0b + M (vs P7a + M),NaN,0.941629,0.000057,0.000722,3,False,0.588769,P7a + M
R1 umbral fijo 0.05,NaN,0.940657,-0.000586,0.001094,1,False,2.692735,NaN
R1 umbral fijo 0.1,NaN,0.940757,-0.000486,0.001081,1,False,2.692735,NaN


In [7]:
# 20 ejemplos al azar entre las sospechosas con umbral 0,05 (de todos los folds externos)
p_etiqueta = np.where(y_np[interna["fila"]] == 1, interna["prob"], 1 - interna["prob"])
sospechosas = interna[p_etiqueta < 0.05].drop_duplicates("fila")
muestra = sospechosas.sample(20, random_state=SEED)
pd.DataFrame({
    "título": crudo.loc[muestra["fila"], "title"].to_numpy(),
    "etiqueta": crudo.loc[muestra["fila"], "condition"].to_numpy(),
    "P(new)": muestra["prob"].round(3).to_numpy(),
    "tipo": crudo.loc[muestra["fila"], "listing_type_id"].to_numpy(),
    "garantía": crudo.loc[muestra["fila"], "warranty"].fillna("").str[:40].to_numpy(),
    "pubs_vendedor": crudo["seller_id"].map(crudo["seller_id"].value_counts()).loc[muestra["fila"]].to_numpy(),
})

,título,etiqueta,P(new),tipo,garantía,pubs_vendedor
0,Wegener El Origen De Los Continentes Y Oceanos Nº 69,new,0.012,bronze,,88
1,Sistema Barral Antipánico Jaque,used,0.977,bronze,,2
2,Recarga Toner Xerox Xc 3220 - 3210 -cod 106r01487,used,0.994,bronze,,2
3,Star Wars - Mon Mothma /tpotf Hasbro Loose,used,0.987,bronze,,2
4,Corolla Xei Manual Tela,new,0.034,gold_premium,,1
5,The Chemical Brothers - Out Of Control ( Sasha Mixes ) Promo,new,0.035,bronze,,12
6,10k La Década Robada - Jorge Lanata - Ed. Planeta,used,0.983,bronze,,2
7,Yegua Zaina De Siete Años,new,0.036,free,,1
8,Los 7 Magnificos Vhs John Sturges Yul Brinner,new,0.027,bronze,,188
9,Cd Pantera The Great Southern Trendkill,new,0.044,bronze,,23


**Lectura de R1.**
- Se sacan 0,7% / 1,5% / 2,7% de las filas de train (umbrales 0,05 / 0,10 / 0,20).
- **Limpiar empeora en todos los umbrales:** −0,0006, −0,0005 y −0,0021. El umbral elegido con 4 folds es **0 (no limpiar) en los 5 folds**.
- **Los 20 ejemplos no parecen errores claros de etiqueta.** Se ven tres grupos:
  - **El modelo se equivoca y la etiqueta está bien:** "Campera … **Sin Usar** …" etiquetada `new` (P(new) = 0,04); "Star Wars … **Loose**" (sin caja) etiquetada `used`.
  - **Etiqueta ambigua, del vendedor:** libros, revistas, CDs y VHS viejos etiquetados `new` por vendedores con muchas publicaciones (El Gráfico 1994, VHS de "Los 7 Magníficos", Bataille, Wegener). "Nuevo" puede querer decir "sin uso" aunque el artículo sea viejo.
  - **Casos donde la condición casi no aplica:** "Yegua Zaina de siete años" (`new`), un Corolla `new`.
- **Conclusión:** las "sospechosas" son sobre todo **casos difíciles o ambiguos, no etiquetas mal puestas**. Sacarlas le quita al modelo justamente los ejemplos de los que aprende a manejar esas zonas, y como el test tiene las mismas ambigüedades (sin limpiar), empeora.

## R2 — Propagación por vendedor

**Qué hace.** Muchos vendedores sin historial en el train tienen **otras publicaciones en el mismo lote sin etiquetas** (en la entrega, el test). No sabemos sus etiquetas, pero sí qué predice el modelo para ellas. Si el modelo está seguro de que las otras 5 publicaciones del vendedor son `new`, es probable que esta también lo sea. Es como el `seller_pct_new`, pero con **predicciones** en lugar de etiquetas.

**Dos etapas:**
1. **Etapa 1:** probabilidad de `new` de cada fila, de un modelo que no la vio. Se usa el HistGB P7a + duración: correlación 0,993 con el modelo final y 5 veces más barato que CatBoost.
2. **Feature:** para cada fila, el **promedio de las predicciones de las OTRAS publicaciones del mismo vendedor en el mismo lote** (`r2_prom`, NaN si no hay ninguna) y cuántas son (`r2_n`).
3. **Etapa 2:** HistGB P7a + duración + esas 2 columnas.

**Cómo se arma en cada lado (para que tenga la misma calidad en train que en test):**

| | Lote | Predicciones de etapa 1 |
|---|---|---|
| Validación (fold externo k) / test en la entrega | el fold k completo / el test | modelo entrenado con los otros 4 folds / con todo el train |
| Train del fold k | el **fold interno** de cada fila (CV interna de 5) | las *out-of-fold* internas: de un modelo que no vio ese lote |

**Por qué no es leakage:**
- la feature de una fila de validación usa **predicciones** de otras filas de validación, hechas por un modelo entrenado sin el fold k. Nunca usa una etiqueta del fold k;
- en el train, la feature de una fila usa predicciones de un modelo que **no vio su lote** (*cross-fitting*), así que no "sabe" las etiquetas de ese lote. Si se usaran predicciones in-sample, serían casi perfectas en train y el modelo de etapa 2 confiaría de más en ellas.

**Desajuste conocido:** el lote de validación tiene 14.000 filas, el lote interno de train ~11.200 y el test 30.000. Con un lote más grande hay más "otras publicaciones" por vendedor, así que `r2_n` tiende a ser mayor en test que en la CV. `r2_prom` (un promedio) debería sufrirlo menos.

In [8]:
def promedio_otras(vendedor, prob, lote):
    # Promedio de prob de las OTRAS filas del mismo vendedor en el mismo lote (NaN si no hay) y cuántas son
    d = pd.DataFrame({"v": np.asarray(vendedor), "l": np.asarray(lote), "p": np.asarray(prob)})
    g = d.groupby(["v", "l"])["p"]
    suma, n = g.transform("sum"), g.transform("size")
    return ((suma - d["p"]) / (n - 1)).where(n > 1).to_numpy(), (n - 1).to_numpy()


def r2_fold(k):
    d = interna[interna["fold_externo"] == k]
    filas_tr, filas_va = d["fila"].to_numpy(), np.where(folds == k)[0]
    X = train.iloc[np.r_[filas_tr, filas_va]].copy()
    prom_tr, n_tr = promedio_otras(train["seller_id"].to_numpy()[filas_tr], d["prob"], d["lote_interno"])
    prom_va, n_va = promedio_otras(train["seller_id"].to_numpy()[filas_va], prob_ref[filas_va], np.zeros(len(filas_va)))
    X["r2_prom"], X["r2_n"] = np.r_[prom_tr, prom_va], np.r_[n_tr, n_va]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = p7a_con([("r2", "passthrough", ["r2_prom", "r2_n"])])
        pipeline.fit(X.iloc[:len(filas_tr)], y.iloc[filas_tr])
        prob = pipeline.predict_proba(X.iloc[len(filas_tr):])[:, 1]
    return filas_va, prob, prom_va, n_va, np.mean(~np.isnan(prom_tr))


inicio = time.time()
salidas_r2 = Parallel(n_jobs=5)(delayed(r2_fold)(k) for k in range(1, 6))
prob_r2, r2_prom_va, r2_n_va = np.empty(len(train)), np.empty(len(train)), np.empty(len(train))
for filas_va, prob, prom_va, n_va, cobertura_tr in salidas_r2:
    prob_r2[filas_va], r2_prom_va[filas_va], r2_n_va[filas_va] = prob, prom_va, n_va
print(f"{(time.time() - inicio) / 60:.1f} min")
print("% de filas con r2_prom (otras del vendedor en el lote): validación",
      np.mean(~np.isnan(r2_prom_va)).round(3), "| train (interno):", np.round([s[4] for s in salidas_r2], 3))

acc_r2 = acc_por_fold(prob_r2)
resultados["R2 propagación por vendedor"] = {"con_P7a": acc_r2.mean(), **comparar(acc_r2, acc_ref), "minutos": (time.time() - inicio) / 60}

# Efecto separado: vendedores SIN historial (ninguna publicación en el train del fold) y con historial
sin_historial = np.zeros(len(train), bool)
for k in range(1, 6):
    filas_va = folds == k
    sin_historial[filas_va] = ~train.loc[filas_va, "seller_id"].isin(set(train.loc[~filas_va, "seller_id"])).to_numpy()
for nombre, filas in [("sin historial", sin_historial), ("sin historial y con otras en el lote", sin_historial & ~np.isnan(r2_prom_va)),
                      ("con historial", ~sin_historial)]:
    a_ref, a_r2 = acc_por_fold(prob_ref, filas), acc_por_fold(prob_r2, filas)
    print(f"{nombre:38s} filas/fold ~{filas.sum() / 5:.0f} | P7a {a_ref.mean():.4f} | R2 {a_r2.mean():.4f} | "
          f"dif {np.mean(a_r2 - a_ref):+.4f} ({int((np.round(a_r2 - a_ref, 10) > 0).sum())}/5)")
pd.DataFrame(resultados).T.round(4)

0.7 min
% de filas con r2_prom (otras del vendedor en el lote): validación 0.53 | train (interno): [0.5   0.505 0.498 0.502 0.504]
sin historial                          filas/fold ~4303 | P7a 0.8923 | R2 0.8928 | dif +0.0005 (2/5)
sin historial y con otras en el lote   filas/fold ~317 | P7a 0.9263 | R2 0.9251 | dif -0.0011 (3/5)
con historial                          filas/fold ~9697 | P7a 0.9630 | R2 0.9628 | dif -0.0001 (2/5)


,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos,referencia
R0a n_vendedores_mismo_precio_cat,0.6526,0.940943,-0.0003,0.000769,2,False,0.64473,P7a
R0a n_publicaciones_mismo_precio_cat,0.6371,0.940943,-0.0003,0.000506,1,False,0.571504,P7a
R0a n_vendedores_mismo_precio_global,0.696043,0.9409,-0.000343,0.00159,2,False,0.634353,P7a
R0a (las 3 juntas),0.7051,0.940943,-0.0003,0.000916,2,False,0.609825,P7a
"R0b (las 3, solo train del fold)",NaN,0.941186,-0.000057,0.001086,2,False,0.628787,P7a
M (5 flags de redondez),NaN,0.941571,0.000329,0.001024,2,False,0.601331,P7a
R0a + M (vs P7a + M),NaN,0.941143,-0.000429,0.001029,2,False,0.598821,P7a + M
R0b + M (vs P7a + M),NaN,0.941629,0.000057,0.000722,3,False,0.588769,P7a + M
R1 umbral fijo 0.05,NaN,0.940657,-0.000586,0.001094,1,False,2.692735,NaN
R1 umbral fijo 0.1,NaN,0.940757,-0.000486,0.001081,1,False,2.692735,NaN


## R3 — Conteos sobre train + test (sin target)

`n_vendedor_total`, `n_categoria_total` y `n_titulo_total`: cuántas publicaciones hay de ese vendedor, de esa categoría y con ese **título normalizado** (minúsculas, sin tildes ni signos, espacios simples), contando **train + test**. Es un *frequency encoder* (`Copia de AP_Categóricas.ipynb`) calculado sobre el universo completo.

**Por qué no es leakage:** solo cuenta filas; **nunca usa `condition`**. Usa features del test, que están disponibles en la competencia. Dentro de la CV, una fila de validación cuenta a las otras filas (de train, de validación y de test), igual que una fila de test en la entrega: las dos se describen igual. El modelo no puede deducir ninguna etiqueta de un conteo.

**Por qué va afuera del `Pipeline`** (a diferencia del conteo de `VendedorEncoder`, que se aprende en el fold): por definición se calcula sobre el universo fijo de 100.000 filas, que es el mismo en la CV y en la entrega.

**Limitación para producción:** es *transductivo*. Depende de tener el test completo de antemano. Una publicación nueva que llega sola no tendría "test" con el cual contarse, y los conteos cambiarían a medida que llegan publicaciones. El conteo del vendedor en train + test ya se probó en `10_vendedor_v2` (V2) sobre un modelo viejo (4/5, no pasó). Acá se re-mide sobre el modelo actual.

In [9]:
def normalizar(titulo):
    t = unicodedata.normalize("NFKD", str(titulo).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^a-z0-9]+", " ", t).split())


universo = pd.DataFrame({"vendedor": np.r_[crudo["seller_id"], crudo_test["seller_id"]],
                         "categoria": np.r_[crudo["category_id"], crudo_test["category_id"]],
                         "titulo": [normalizar(t) for t in np.r_[crudo["title"], crudo_test["title"]]]})
for col, nueva in [("vendedor", "n_vendedor_total"), ("categoria", "n_categoria_total"), ("titulo", "n_titulo_total")]:
    train[nueva] = universo[col].map(universo[col].value_counts()).to_numpy()[:len(train)]
print(train[["n_vendedor_total", "n_categoria_total", "n_titulo_total"]].describe().round(1))
print("% de filas de train con título repetido en train + test:", (train["n_titulo_total"] > 1).mean().round(4))

R3 = {"R3a n_vendedor_total": ["n_vendedor_total"], "R3b n_categoria_total": ["n_categoria_total"],
      "R3c n_titulo_total": ["n_titulo_total"], "R3 (las 3 juntas)": ["n_vendedor_total", "n_categoria_total", "n_titulo_total"]}
for nombre, cols in R3.items():
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sola = (cross_validate(HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED),
                               train[cols], y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"].mean() if len(cols) == 1 else np.nan)
        acc = cross_validate(p7a_con([("r3", "passthrough", cols)]), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
    resultados[nombre] = {"señal_sola": sola, "con_P7a": acc.mean(), **comparar(acc, acc_ref), "minutos": (time.time() - inicio) / 60}
    print(nombre, round(acc.mean(), 4), comparar(acc, acc_ref))

       n_vendedor_total  n_categoria_total  n_titulo_total
count           70000.0            70000.0         70000.0
mean               46.6              336.2             1.2
std               124.8              975.4             2.6
min                 1.0                1.0             1.0
25%                 1.0               10.0             1.0
50%                 6.0               33.0             1.0
75%                32.0              142.0             1.0
max               933.0             4593.0            78.0
% de filas de train con título repetido en train + test: 0.0211


R3a n_vendedor_total 0.9416 {'dif_media': np.float64(0.00034285716), 'dif_desvio': np.float64(0.000690164131381006), 'folds_gana': 3, 'pasa': False}


R3b n_categoria_total 0.9411 {'dif_media': np.float64(-0.00017142856000000004), 'dif_desvio': np.float64(0.0007647862261269292), 'folds_gana': 1, 'pasa': False}


R3c n_titulo_total 0.9411 {'dif_media': np.float64(-0.00011428571999999998), 'dif_desvio': np.float64(0.00028428210392538887), 'folds_gana': 1, 'pasa': False}


R3 (las 3 juntas) 0.9412 {'dif_media': np.float64(-4.336808689942018e-20), 'dif_desvio': np.float64(0.0007928249563608425), 'folds_gana': 3, 'pasa': False}


## Tabla de la ronda (contra HistGB P7a + duración = 0,9412)

In [10]:
tabla = pd.DataFrame(resultados).T[["señal_sola", "con_P7a", "dif_media", "dif_desvio", "folds_gana", "pasa", "minutos"]]
tabla.to_csv(os.path.join(carpeta_lab, "ideas_finales.csv"))
pd.DataFrame({"fold": folds, "y": y_np, "r2": prob_r2, **{f"r1_{u}": prob_r1[u] for u in umbrales}}).to_csv(
    os.path.join(carpeta_lab, "oof_ideas_finales.csv"), index=False)
tabla.round(4)

,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos
R0a n_vendedores_mismo_precio_cat,0.6526,0.940943,-0.0003,0.000769,2,False,0.64473
R0a n_publicaciones_mismo_precio_cat,0.6371,0.940943,-0.0003,0.000506,1,False,0.571504
R0a n_vendedores_mismo_precio_global,0.696043,0.9409,-0.000343,0.00159,2,False,0.634353
R0a (las 3 juntas),0.7051,0.940943,-0.0003,0.000916,2,False,0.609825
"R0b (las 3, solo train del fold)",NaN,0.941186,-0.000057,0.001086,2,False,0.628787
M (5 flags de redondez),NaN,0.941571,0.000329,0.001024,2,False,0.601331
R0a + M (vs P7a + M),NaN,0.941143,-0.000429,0.001029,2,False,0.598821
R0b + M (vs P7a + M),NaN,0.941629,0.000057,0.000722,3,False,0.588769
R1 umbral fijo 0.05,NaN,0.940657,-0.000586,0.001094,1,False,2.692735
R1 umbral fijo 0.1,NaN,0.940757,-0.000486,0.001081,1,False,2.692735


## Conclusión de la ronda — nada pasa, se cierran los experimentos

| Idea | Mejor variante | Con P7a (ref. 0,9412) | Diferencia (folds) |
|---|---|---|---|
| R0 precio compartido (train + test) | las 3 juntas (señal sola **0,705**) | 0,9409 | −0,0003 (2/5) |
| R0b precio compartido (solo train del fold) | las 3 | 0,9412 | −0,0001 (2/5) |
| R0 controlando redondez (ref. P7a + M = 0,9416) | R0b + M | 0,9416 | +0,0001 (3/5) |
| M redondez (bloque M) | 5 flags | 0,9416 | +0,0003 (2/5) |
| R1 limpieza de etiquetas | umbral elegido en 4 folds = 0 (no limpiar) | 0,9412 | 0 (0/5); umbrales fijos: −0,0005 a −0,0021 |
| R2 propagación por vendedor | — | 0,9413 | +0,0000 (2/5) |
| R3 conteos train + test | vendedor | 0,9416 | +0,0003 (3/5) |

- **R0:** es la feature con **más señal sola de todo el TP** (0,705; la mejor de la 18 era 0,632) y no es solo redondez: entre precios **no** redondos, compartido = 33% `new` contra 71% único. Pero **no suma**: lo que dice ("precio de mercado de un usado") ya lo dicen el vendedor (particular contra tienda), el tipo de publicación y el texto. Contar sobre train + test (R0a) no da ventaja frente a contar solo en el fold (R0b).
- **R1:** las sospechosas son casos ambiguos, no errores de etiqueta (ver ejemplos). Limpiar empeora.
- **R2:** solo el 53% de las filas de validación tiene otra publicación del mismo vendedor en su lote. En los vendedores **sin historial** cambia +0,0005 (2/5); en los sin historial **con** otras en el lote (~317 filas por fold), −0,0011. El modelo ya usa el contenido de esas publicaciones, y el promedio de predicciones de las vecinas no agrega información nueva.
- **R3:** los conteos sobre train + test (vendedor, categoría, título) quedan en el ruido, igual que V2 en `10_vendedor_v2`.

**Decisión: no se cambia nada y se cierran los experimentos.** Modelo final = P7c + duración (CV 0,9420, público 0,94247). Sin confirmación en el promedio (no hubo candidatas), sin `features.py` y sin submission. Resultados en `.cache_laboratorio/ideas_finales.csv`, OOF en `oof_ideas_finales.csv`, OOF internas en `oof_interna_p7a_duracion.csv`.